# IMDB RNN Sentiment

Train a simple RNN model on the IMDB movie review dataset, save the model and preprocessing artifacts, and run a quick prediction sanity check.

In [2]:
%pip install numpy tensorflow

  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl (350.9 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\hares\\OneDrive\\Desktop\\tekWorks\\fullstack_data_scientist\\Deep_Learning\\RNN\\.venv\\Lib\\site-packages\\tensorflow\\include\\external\\envoy_api\\envoy\\extensions\\load_balancing_policies\\client_side_weighted_round_robin\\v3\\client_side_weighted_round_robin.upb.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [1]:
import json
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

NUM_WORDS = 10000
MAX_LEN = 250
INDEX_FROM = 3
ARTIFACT_DIR = "artifacts"
MODEL_PATH = os.path.join(ARTIFACT_DIR, "imdb_rnn.keras")
WORD_INDEX_PATH = os.path.join(ARTIFACT_DIR, "word_index.json")
META_PATH = os.path.join(ARTIFACT_DIR, "meta.json")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [2]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=NUM_WORDS, index_from=INDEX_FROM)

x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding="post", truncating="post")
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding="post", truncating="post")

raw_word_index = imdb.get_word_index()
word_index = {k: (v + INDEX_FROM) for k, v in raw_word_index.items()}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<OOV>"] = 2

with open(WORD_INDEX_PATH, "w", encoding="utf-8") as f:
    json.dump(word_index, f)

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump({"num_words": NUM_WORDS, "max_len": MAX_LEN}, f)

x_train.shape, x_test.shape

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


((25000, 250), (25000, 250))

In [3]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(NUM_WORDS, 128, input_length=MAX_LEN),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

model.save(MODEL_PATH)
MODEL_PATH

c:\Users\hares\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 64s 393ms/step - accuracy: 0.7258 - loss: 0.5291 - val_accuracy: 0.8482 - val_loss: 0.3742
Epoch 2/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 55s 351ms/step - accuracy: 0.8828 - loss: 0.3050 - val_accuracy: 0.8672 - val_loss: 0.3344
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 56s 355ms/step - accuracy: 0.9169 - loss: 0.2309 - val_accuracy: 0.8626 - val_loss: 0.3537


'artifacts\\imdb_rnn.keras'

In [4]:
def encode_review(text, word_index, num_words):
    words = text.lower().split()
    encoded = [word_index["<START>"]]
    for word in words:
        idx = word_index.get(word, word_index["<OOV>"])
        if idx >= num_words:
            idx = word_index["<OOV>"]
        encoded.append(idx)
    return encoded

sample_text = "This movie was fantastic, I loved the story and acting."
sample_encoded = encode_review(sample_text, word_index, NUM_WORDS)
sample_padded = pad_sequences([sample_encoded], maxlen=MAX_LEN, padding="post", truncating="post")
score = float(model.predict(sample_padded, verbose=0)[0][0])
label = "good" if score >= 0.5 else "bad"
score, label

(0.6743781566619873, 'good')